In [2]:
import torch
torch.cuda.is_available()

True

In [43]:
torch.__version__

'2.7.1+cu118'

In [46]:
torch.cuda.get_device_capability()

(8, 6)

In [1]:
!nvidia-smi

Sat Oct 18 09:41:10 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.57                 Driver Version: 581.57         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3080 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   53C    P0            751W /  129W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# MFU - 김정인 강사(jikim@imguru.co.kr)
# 밑바닥부터 시작하는 딥러닝
# 1~5권까지 있음(컨볼루션-LLM(LSTM)-프레임워크만들기-...)

# c언어의 타입 = 총 4가지임
# char: 1byte - 8bit
# int: 4byte - 32bit
# float
# double
# 이외에는 조금 다른 것들임

# char a = 200 -> -56

# 뜬금없는 ppt 복사법: 사각형을 만들고 ctrl+d로 복사,
# 이동할 곳으로 복사본 이동 후 ctrl+d 를 누르면 이전패턴대로 복사됨(이전에 최종복사본 위치만큼 이동 복사)

# 고정소수점의 문제 
# 1. 매우 큰 수 표현 불가
# 2. 매우 정밀한 수를 표현할 수 없음

# 부동(부동)소수점 방식(지수표현법)은 위의 문제를 해결함
# 새로운 문제점
# 1. 지수에도 음수가 있음
# 2. 가수부의 첫번째 숫자는 항상 1이다

In [4]:
# 라이브러리 설치
!pip install calflops


   ---------------------------------------- 0/2 [accelerate]
   -------------------- ------------------- 1/2 [calflops]
   ---------------------------------------- 2/2 [calflops]



In [ ]:
import torch
from calflops import calculate_flops

def parse_flop_string(s):
    return float(s.strip().split()[0])

model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', weights='ResNet18_Weights.DEFAULT')

flops, macs, params = calculate_flops(
    model=model,
    input_shape=(1, 3, 224, 224),
    print_results=False
)

print(f"FLOPs: {parse_flop_string(flops):.2f}G, MACs: {parse_flop_string(macs):.2f}G")

c:\Users\devchoi\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloading: "https://github.com/pytorch/vision/zipball/v0.10.0" to C:\Users\devchoi/.cache\torch\hub\v0.10.0.zip
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\devchoi/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:05<00:00, 9.25MB/s]


FLOPs: 3.64G, MACs: 1.81G


In [2]:
import torch, time, os

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    os.system("nvidia-smi | head -n 20")

CUDA available: True
GPU: NVIDIA GeForce RTX 3080 Ti Laptop GPU


In [ ]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, 3, stride=2, padding=1) # x = (1,3,224,224)
                                                             # w = (32,3,3,3)
                                                             # out = (1,32,112,112)

                                                            #  (N+2P-F)/S + 1
        self.fc = nn.Linear(32 * 112 * 112, 10) # (1,32*112*112)(32*112*112,10) => (1,10)
    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = SimpleCNN().cuda().eval()

In [4]:
# FLOPs = 2 * H * W * Cin * Cout * Kh * Kw
H, W, Cin, Cout, Kh, Kw = 224, 224, 3, 32, 3, 3
flops_conv = 2 * H/2 * W/2 * Cin * Cout * Kh * Kw  # stride=2 → H/2,W/2
flops_fc = 2 * (32 * 112 * 112) * 10
total_flops = flops_conv + flops_fc
print(f"총 FLOPs: {total_flops/1e9:.3f} GFLOPs")

총 FLOPs: 0.030 GFLOPs


In [5]:
x = torch.randn(32, 3, 224, 224).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
y = torch.randint(0, 10, (32,)).cuda()

torch.cuda.synchronize()
start = time.time()

for _ in range(50):  # 50 iterations
    optimizer.zero_grad()
    out = model(x)
    loss = criterion(out, y)
    loss.backward()
    optimizer.step()

torch.cuda.synchronize()
end = time.time()
train_time = (end - start) / 50
print(f"평균 반복당 학습 시간: {train_time:.4f} 초")

평균 반복당 학습 시간: 0.0225 초


In [6]:
gpu_theoretical_flops = 19.5e12  # A100 기준 (FP32)
mfu = (total_flops / train_time) / gpu_theoretical_flops * 100
print(f"🔹 MFU(Model FLOPs Utilization): {mfu:.2f}%")

🔹 MFU(Model FLOPs Utilization): 0.01%


In [7]:
!nvidia-smi --query-gpu=utilization.gpu,utilization.memory,memory.used --format=csv

utilization.gpu [%], utilization.memory [%], memory.used [MiB]
0 %, 0 %, 530 MiB


병목구간 찾기는 내일할 예정

In [8]:
import torch
import torch.nn as nn

# FC (Linear) 레이어 정의
N_in, N_out = 4, 3
fc = nn.Linear(N_in, N_out, bias=False)

# 입력 데이터 (배치 크기 1)
x = torch.randn(1, N_in)
y = fc(x)

print("입력 크기:", x.shape)
print("출력 크기:", y.shape)

# FLOPs 계산
flops = 2 * N_in * N_out
print(f"이론적 FLOPs: {flops} 회 연산 (곱셈 + 덧셈 포함)")

입력 크기: torch.Size([1, 4])
출력 크기: torch.Size([1, 3])
이론적 FLOPs: 24 회 연산 (곱셈 + 덧셈 포함)


In [9]:
import torch
import torch.nn as nn
import numpy as np

# 입력 및 커널 정의
x = torch.randn(1, 3, 4, 4)   # (배치, 채널, 높이, 너비)
conv = nn.Conv2d(in_channels=3, out_channels=1, kernel_size=3, stride=1, padding=0)

# 연산 수행
y = conv(x)
print("출력 크기:", y.shape)

# FLOPs 계산 공식 적용
H_out, W_out = y.shape[2], y.shape[3]
K_h, K_w = conv.kernel_size
C_in, C_out = conv.in_channels, conv.out_channels

flops = 2 * H_out * W_out * K_h * K_w * C_in * C_out
print(f"이론적 FLOPs: {flops} 회 연산")

출력 크기: torch.Size([1, 1, 2, 2])
이론적 FLOPs: 216 회 연산


In [10]:
import torch
import torch.nn as nn

# 간단한 모델 정의
model = nn.Sequential(
    nn.Conv2d(3, 16, 3, stride=1, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 32, 3, stride=1, padding=1),
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(32 * 32 * 32, 10)
)

x = torch.randn(1, 3, 32, 32)

# MACs 계산 예시
H_out, W_out = 32, 32
K_h, K_w = 3, 3
C_in, C_out = 3, 16

conv1_macs = H_out * W_out * K_h * K_w * C_in * C_out
print(f"Conv1 MACs: {conv1_macs / 1e6:.2f} MMACs")

Conv1 MACs: 0.44 MMACs


# 아래 코드는 코랩에서 잘됨

In [52]:
312_000

312000

In [ ]:
import torch
import torch.profiler as profiler

# [1] 간단한 테스트용 모델 및 입력 데이터
model = torch.nn.Linear(1024, 1024).cuda()
input_data = torch.randn(16, 1024).cuda()

# [2] 프로파일링용 기본 설정값
batch_size = 16
in_feature = 1024
out_feature = 1024
flops_for_forward = 
iterations = 50   # 총 반복 횟수

# [3] torch.profiler로 CPU/GPU 시간 측정
with profiler.profile(
    activities=[profiler.ProfilerActivity.CPU, profiler.ProfilerActivity.CUDA],
    record_shapes=True
) as prof:
    for i in range(iterations):
        output = model(input_data)
        loss = output.sum()
        loss.backward()
        torch.cuda.synchronize()   # GPU 연산 완료 대기 (정확한 시간 측정용)

# [4] 프로파일링 결과에서 평균 수행시간 계산
events = prof.key_averages()   # 개별 연산별 집계
print(events)
total_cpu_time = sum([e.self_cpu_time_total for e in events]) / 1e6  # (ms → s) # 해당코드는 cpu를 중점으로 확인함
time_per_iter = total_cpu_time / iterations
print(f"🕒 반복 1회당 평균 수행시간: {time_per_iter:.6f} 초")

# [5] 처리량(Throughput) 계산
throughput = batch_size / time_per_iter
print(f"⚡ 처리량(Throughput): {throughput:.2f} 샘플/초")

# [6] FLOPs 및 GPU 이론 성능 설정 - 아래의 두가지 샘플 다 틀린듯
# flops_per_sample = 1e6  # 예시: 1 GFLOP / 샘플
flops_per_sample = 1024*1024  # 이미지 크기로 잡아주자
total_flops = flops_per_sample * batch_size

# NVIDIA A100 FP32 성능 (19.5 TFLOPs)
# gpu_peak_flops = 19.5 * 1e12  # FP32 기준
gpu_peak_flops = 18.7 * 1e12  # rtx3080ti laptop FP32 기준
# 만약 FP16 혼합정밀도(Half precision)라면 → 312 * 1e12 로 변경

# [7] MFU 계산
mfu = (total_flops / time_per_iter) / gpu_peak_flops
print(f"🔥 MFU (Model FLOPs Utilization): {mfu:.2%}")

prof.export_chrome_trace("trace.json")


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::linear         2.31%     450.600us        35.39%       6.904ms     138.086us            50  
                                                aten::t         4.76%     928.500us         8.72%       1.701ms       8.506us           200  
                                        aten::transpose         2.88%     561.500us         3.96%     772.700us       3.864us           200  
                                       aten::as_strided         1.97%     385.100us         1.97%     385.100us       1.284us           300  
      

In [48]:
1024*1024

1048576

# 아래는 수정코드-gem_ver

In [40]:
import torch
import time

# [1] 간단한 테스트용 모델 및 입력 데이터
# 💡 만약 FP16 (혼합정밀도)을 테스트하고 싶다면, model.half() 및 input_data.half()를 사용하세요.
model = torch.nn.Linear(1024, 1024).cuda()
input_data = torch.randn(16, 1024).cuda()

# [2] 기본 설정값
batch_size = 16
iterations = 50 # 측정에 사용할 총 반복 횟수
input_size = 1024
output_size = 1024

# [3] Warm-up (정확한 측정을 위해 최소 10회 이상 수행)
for _ in range(10):
    output = model(input_data)
    loss = output.sum()
    loss.backward()

# [4] time.time()으로 GPU 시간 측정 (torch.profiler 대체)
torch.cuda.synchronize() # Warm-up 완료 대기
start_time = time.time()

for i in range(iterations):
    output = model(input_data)
    loss = output.sum()
    loss.backward()

torch.cuda.synchronize() # GPU 연산 완료 대기 (정확한 시간 측정용)
end_time = time.time()

# [5] 프로파일링 결과에서 평균 수행시간 계산
total_time = end_time - start_time
time_per_iter = total_time / iterations
print(f"🕒 반복 1회당 평균 수행시간: {time_per_iter:.6f} 초")

# [6] 처리량(Throughput) 계산
throughput = batch_size / time_per_iter
print(f"⚡ 처리량(Throughput): {throughput:.2f} 샘플/초")

# [7] FLOPs 및 GPU 이론 성능 설정 (수정됨)
# 순방향 (FW) FLOPS: 2 * Batch Size * Input Size * Output Size
flops_forward = 2 * batch_size * input_size * output_size

# 총 FLOPS (순방향 + 역방향): 순방향의 약 3배로 근사
total_flops_per_iteration = flops_forward * 3

print(f"ℹ️ 반복 1회당 계산된 FLOPS: {total_flops_per_iteration / 1e6:.2f} MFLOPs")


# NVIDIA RTX 3080 Ti Laptop FP32 성능 (약 18.7 TFLOPs)
# 💡 만약 FP16 혼합정밀도라면 해당 GPU의 FP16 이론 성능을 사용해야 합니다.
gpu_peak_flops = 18.7 * 1e12 # TFLOPs → FLOPs

# [8] MFU 계산
actual_flops_per_second = total_flops_per_iteration / time_per_iter
mfu = actual_flops_per_second / gpu_peak_flops
print(f"🔥 MFU (Model FLOPs Utilization): {mfu:.2%}")

# 참고: trace.json 생성은 time.time() 방식으로 변경했기 때문에 주석 처리했습니다.
# prof.export_chrome_trace("trace.json")

🕒 반복 1회당 평균 수행시간: 0.001843 초
⚡ 처리량(Throughput): 8683.01 샘플/초
ℹ️ 반복 1회당 계산된 FLOPS: 100.66 MFLOPs
🔥 MFU (Model FLOPs Utilization): 0.29%


# 강사님 수정본

In [53]:
import torch
from torch.profiler import profile, record_function, ProfilerActivity

# ---------------------------------------------------
# 1️⃣ 모델 및 입력 정의
# ---------------------------------------------------
model = torch.nn.Linear(1024, 1024).cuda()
input_data = torch.randn(16, 1024).cuda()

# ---------------------------------------------------
# 2️⃣ FLOPs 계산 (이론값)
# ---------------------------------------------------
batch_size = 16
in_features = 1024
out_features = 1024
flops_per_forward = 2 * batch_size * in_features * out_features  # multiply + add
print(f"[INFO] Estimated FLOPs per forward: {flops_per_forward/1e6:.2f} MFLOPs")

# ---------------------------------------------------
# 3️⃣ Torch Profiler 설정
# ---------------------------------------------------
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    with_flops=True,
    profile_memory=True
) as prof:
    with record_function("linear_forward"):
        output = model(input_data)

# ---------------------------------------------------
# 4️⃣ 프로파일 결과 요약
# ---------------------------------------------------
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=5))

# ---------------------------------------------------
# 5️⃣ GPU 실행 시간과 FLOPs 추출 (PyTorch 2.x 대응)
# ---------------------------------------------------
total_cuda_time = 0.0
total_flops = 0.0

for evt in prof.key_averages():
    # cuda_time_total은 μs 단위 (없으면 0으로 처리)
    total_cuda_time += getattr(evt, "cuda_time_total", 0.0)
    if hasattr(evt, "flops") and evt.flops is not None:
        total_flops += evt.flops

print("total_flops=", total_flops)

# μs → s 변환
total_cuda_time /= 1e6

print(f"[INFO] Total CUDA time: {total_cuda_time*1e3:.3f} ms")
print(f"[INFO] FLOPs measured by profiler: {total_flops/1e6:.2f} MFLOPs")

# ---------------------------------------------------
# 6️⃣ Throughput & MFU 계산
# ---------------------------------------------------
throughput_gflops = (total_flops / total_cuda_time) / 1e9 if total_cuda_time > 0 else 0.0

# 예: A100 FP16 기준 Peak = 312,000 GFLOPs
gpu_peak_flops = 312_000
mfu = (throughput_gflops / gpu_peak_flops) * 100

print(f"[RESULT] Throughput: {throughput_gflops:.2f} GFLOPs/s")
print(f"[RESULT] MFU (Model FLOPs Utilization): {mfu:.4f}%")

[INFO] Estimated FLOPs per forward: 33.55 MFLOPs
--------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  Total MFLOPs  
--------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
      linear_forward        36.07%     522.200us       100.00%       1.448ms       1.448ms           0 b           0 b           0 b     -64.00 Kb             1            --  
        aten::linear        39.86%     577.100us        63.93%     925.500us     925.500us           0 b           0 b      64.00 Kb           0 b             1            --  
             aten::t         2.39%      34.600us         3.90%   

# 강사님 수정본2

In [54]:
import torch
from torch.profiler import profile, record_function, ProfilerActivity

# ---------------------------------------------------
# 1️⃣ 모델 및 입력 정의
# ---------------------------------------------------
model = torch.nn.Linear(1024, 1024).cuda()
input_data = torch.randn(16, 1024).cuda()

# ---------------------------------------------------
# 2️⃣ FLOPs 계산 (이론값)
# ---------------------------------------------------
batch_size = 16
in_features = 1024
out_features = 1024
flops_per_forward = 2 * batch_size * in_features * out_features  # multiply + add
print(f"[INFO] Estimated FLOPs per forward: {flops_per_forward/1e6:.2f} MFLOPs")

# ---------------------------------------------------
# 3️⃣ FLOPs 측정 (Profiler)
# ---------------------------------------------------
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
             with_flops=True, record_shapes=True) as prof:
    with record_function("linear_forward"):
        output = model(input_data)

total_flops = sum([evt.flops for evt in prof.key_averages() if hasattr(evt, "flops") and evt.flops is not None])
print(f"[INFO] FLOPs measured by profiler: {total_flops/1e6:.2f} MFLOPs")

# ---------------------------------------------------
# 4️⃣ CUDA 실행 시간 측정 (정확한 방식)
# ---------------------------------------------------
torch.cuda.synchronize()
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

# 반복 횟수 (짧은 연산 보정용)
num_iter = 10000

start_event.record()
for _ in range(num_iter):
    _ = model(input_data)
end_event.record()

torch.cuda.synchronize()
elapsed_time_ms = start_event.elapsed_time(end_event)  # ms 단위
avg_time_s = (elapsed_time_ms / num_iter) / 1000.0

print(f"[INFO] Avg forward time per iteration: {avg_time_s*1e3:.4f} ms")

# ---------------------------------------------------
# 5️⃣ Throughput & MFU 계산
# ---------------------------------------------------
throughput_gflops = (flops_per_forward / avg_time_s) / 1e9  # GFLOPs/s
gpu_peak_flops = 312_000  # A100 FP16 기준 (GFLOPs)
mfu = (throughput_gflops / gpu_peak_flops) * 100

print(f"[RESULT] Throughput: {throughput_gflops:.2f} GFLOPs/s")
print(f"[RESULT] MFU (Model FLOPs Utilization): {mfu:.4f}%")

[INFO] Estimated FLOPs per forward: 33.55 MFLOPs
[INFO] FLOPs measured by profiler: 33.55 MFLOPs
[INFO] Avg forward time per iteration: 0.0660 ms
[RESULT] Throughput: 508.10 GFLOPs/s
[RESULT] MFU (Model FLOPs Utilization): 0.1629%


In [55]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Jul_16_20:06:48_Pacific_Daylight_Time_2025
Cuda compilation tools, release 13.0, V13.0.48
Build cuda_13.0.r13.0/compiler.36260728_0


In [58]:
import torch
print(torch.version.cuda)
print(torch.backends.cudnn.version())

11.8
90100


# torch 2.9.0으로 변경 후

In [1]:
import torch
print(torch.version.cuda)
print(torch.backends.cudnn.version())

13.0
91200
